<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/SEM_CNN_30x_7_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 광섬유 SEM CNN — 단계별 실행

위에서 아래 순서로 실행하세요. Colab에서 T4 GPU를 권장합니다. 중간 단계부터 재실행하려면 그 이전 단계가 완료되어 있어야 합니다. 마지막 단계 완료 후에는 임시 데이터가 삭제됩니다.

## 1. Google Drive 연결 및 라이브러리

Google Drive를 연결하고 학습에 필요한 라이브러리를 불러옵니다.

In [1]:
# -*- coding: utf-8 -*-
r"""SEM 분류기. Colab: !python SEM_CNN_3epoch.py
Windows: python SEM_CNN_3epoch.py --data-dir "D:\path\Fiber_Category_Renamed_PNG"
필요 패키지: torch torchvision numpy pandas pillow scikit-learn tqdm
전체 %는 단계별 가중치에 기반한 진행률이며 예상 소요시간 비율이 아닙니다.
"""
import sys, argparse, tempfile
from pathlib import Path
parser=argparse.ArgumentParser(description='SEM CNN: 3 epoch')
parser.add_argument('--data-dir', help='Fiber_Category_Renamed_PNG 폴더 경로')
parser.add_argument('--output-dir', help='결과를 저장할 상위 폴더')
args=parser.parse_known_args()[0]
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
from pathlib import Path
import os, random, hashlib, shutil, json, time
import numpy as np, pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


Mounted at /content/drive


## 2. 학습 설정

3 epoch, 원본 1장당 증강 29장으로 학습용 사진을 총 30배로 만듭니다.

In [2]:
SEED=42; EPOCHS=20; BATCH_SIZE=32; IMG_SIZE=224
AUG_PER_IMAGE=29  # 원본 포함 총 30장
CLASSES=['bending','cleaving','fracture']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('장치:',device)
if device.type=='cpu': print('GPU가 없어 시간이 오래 걸릴 수 있습니다. 런타임 설정에서 T4 GPU를 선택하세요.')


장치: cpu
GPU가 없어 시간이 오래 걸릴 수 있습니다. 런타임 설정에서 T4 GPU를 선택하세요.


## 3. 진행률과 저장 경로 설정

원본 폴더와 결과 폴더를 지정합니다. 전체 진행률은 단계별 가중치이며 시간 비율은 아닙니다.

In [3]:
# 전체 진행률은 단계별 가중치를 사용한 작업량 지표이며 시간 예측이 아닙니다.
def progress(items, start, end, title, total=None):
    total = len(items) if total is None else int(total)
    print(f"[전체 {start:.1f}%] {title} 시작 (총 {total})", flush=True)
    next_mark=0
    for i, item in enumerate(items,1):
        yield item
        pct=start+(end-start)*i/max(total,1)
        phase=100*i/max(total,1)
        if phase >= next_mark or i==total:
            print(f"[전체 {pct:.1f}%] {title}: {i}/{total} ({phase:.0f}%)",flush=True)
            next_mark=(int(phase)//10+1)*10
    if not total: print(f"[전체 {end:.1f}%] {title}: 처리할 항목 없음",flush=True)
if args.data_dir:
    DATA_DIR=Path(args.data_dir).expanduser()
elif IN_COLAB:
    DATA_DIR=Path('/content/drive/MyDrive/Fiber_Category_Renamed_PNG')
else:
    raise SystemExit('PC에서는 --data-dir 로 데이터 폴더 경로를 지정하세요.')
assert DATA_DIR.is_dir(), f'폴더 없음: {DATA_DIR}'
out=Path(args.output_dir).expanduser() if args.output_dir else (Path('/content/drive/MyDrive/SEM_CNN_3epoch') if IN_COLAB else DATA_DIR.parent/'SEM_CNN_3epoch')
RUN=out/time.strftime('%Y%m%d_%H%M%S')
RUN.mkdir(parents=True,exist_ok=False)
LOCAL=Path(tempfile.mkdtemp(prefix='sem_cnn_'))
print('원본 폴더:',DATA_DIR,'결과 저장:',RUN,flush=True)


원본 폴더: /content/drive/MyDrive/Fiber_Category_Renamed_PNG 결과 저장: /content/drive/MyDrive/SEM_CNN_3epoch/20261108_081105


## 4. 원본 이미지 확인

폴더 라벨과 HCF 여부를 확인합니다. 중복 이미지도 모두 포함하며 열 수 없는 파일은 건너뜁니다.

In [4]:
# 이미지 읽기 확인 및 폴더별 라벨 부여: 동일 사진도 모두 포함
rows = []
files = []
for label in CLASSES:
    root = DATA_DIR / label
    assert root.is_dir(), f'폴더 없음: {root}'
    files.extend((label, p) for p in sorted(root.rglob('*'))
                 if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff'})

for label, p in progress(files, 0, 15, '원본 확인'):
    root = DATA_DIR / label
    try:
        with Image.open(p) as im:
            ImageOps.exif_transpose(im).convert('RGB').load()
    except Exception as e:
        print('읽기 실패:', p, e)
        continue
    rows.append({
        'path': str(p),
        'label': label,
        'hcf': any(x.lower() == 'hcf' for x in p.relative_to(root).parts[:-1])
               or 'hcf' in p.stem.lower(),
    })

df = pd.DataFrame(rows)
assert len(df) > 0, '이미지가 없습니다.'
print(pd.crosstab(df.label, df.hcf))
print('사용할 이미지 전체:', len(df))


[전체 0.0%] 원본 확인 시작 (총 678)
[전체 0.0%] 원본 확인: 1/678 (0%)
[전체 1.5%] 원본 확인: 68/678 (10%)
[전체 3.0%] 원본 확인: 136/678 (20%)
[전체 4.5%] 원본 확인: 204/678 (30%)
[전체 6.0%] 원본 확인: 272/678 (40%)
[전체 7.5%] 원본 확인: 339/678 (50%)
[전체 9.0%] 원본 확인: 407/678 (60%)
[전체 10.5%] 원본 확인: 475/678 (70%)
[전체 12.0%] 원본 확인: 543/678 (80%)
[전체 13.5%] 원본 확인: 611/678 (90%)
[전체 15.0%] 원본 확인: 678/678 (100%)
hcf       False  True 
label                 
bending     146      0
cleaving    301    105
fracture    118      8
사용할 이미지 전체: 678


## 5. 학습·검증·테스트 분할

원본을 파단 유형 기준으로 70:20:10으로 먼저 분할합니다. 각 원본과 그 증강본은 같은 그룹에만 들어갑니다. 동일 사진이나 동일 시편의 다른 사진은 자동으로 묶지 않으므로 분할 전 확인이 필요합니다. HCF 표본이 적으면 일부 분할에 없을 수 있습니다.

In [5]:
# 유형을 기준으로 70/20/10 분할. HCF 표본이 작으면 각 분할에 없을 수 있음.
idx=np.arange(len(df)); y=df.label
train,tmp=train_test_split(idx,test_size=.30,random_state=SEED,stratify=y)
val,test=train_test_split(tmp,test_size=1/3,random_state=SEED,stratify=y.iloc[tmp])
df['split']=''; df.loc[train,'split']='train'; df.loc[val,'split']='val'; df.loc[test,'split']='test'
df.to_csv(RUN/'manifest.csv',index=False,encoding='utf-8-sig')
print(pd.crosstab([df['split'],df.label],df.hcf))


hcf             False  True 
split label                 
test  bending      15      0
      cleaving     34      7
      fracture     12      0
train bending     102      0
      cleaving    203     81
      fracture     80      8
val   bending      29      0
      cleaving     64     17
      fracture     26      0


## 6. 분할별 이미지 증강 및 Drive 저장

각 분할에서 원본 1장과 증강 29장을 저장합니다. Train·Validation·Test는 원본 분할을 유지합니다. 검증·테스트에서는 증강본 평가와 별도로 원본만의 테스트 성능도 출력합니다. 증강으로 생긴 이미지는 독립 시편이 아닙니다.

In [6]:
# 원본 분할별로 증강해 Drive에 저장
from torchvision.transforms import functional as TF, InterpolationMode
resize = transforms.Resize((IMG_SIZE, IMG_SIZE))
saved = RUN / 'augmented_dataset'

def gaussian_noise(im, std=0.025):
    # 단일 강도 노이즈를 RGB에 공통 적용하여 SEM 흑백 성질 유지
    tensor = TF.to_tensor(im)
    noise = torch.randn(1, tensor.shape[1], tensor.shape[2]) * std
    return TF.to_pil_image((tensor + noise).clamp(0, 1))

def augment_variant(im, k):
    # 초기 10장과 이후 무작위 변환 조합으로 다양성을 높입니다.
    import random
    from torchvision.transforms import functional as VF
    x = im.copy()
    if k == 0: return VF.hflip(x)
    if k == 1: return VF.vflip(x)
    if k in (2,3,4): return VF.rotate(x, (k-1)*90)
    if k in (5,6):
        return VF.affine(x, angle=0, translate=[0,0], scale=(0.9 if k==5 else 1.1), shear=[0.0,0.0], interpolation=InterpolationMode.BILINEAR)
    if k in (7,8): return gaussian_noise(x, 0.015 if k==7 else 0.025)
    if random.random() < 0.5: x = VF.hflip(x)
    if random.random() < 0.5: x = VF.vflip(x)
    angle = random.choice([0,90,180,270])
    x = VF.rotate(x, angle)
    scale = random.uniform(0.92, 1.08)
    x = VF.affine(x, angle=0, translate=[0,0], scale=scale, shear=[0.0,0.0], interpolation=InterpolationMode.BILINEAR)
    if random.random() < 0.7: x = gaussian_noise(x, random.uniform(0.008,0.025))
    return x

for i,row in progress(df.iterrows(),15,45,'분할별 증강 및 저장',total=len(df)):
    with Image.open(row.path) as im:
        im = resize(ImageOps.exif_transpose(im).convert('RGB'))
    folder = saved / row['split'] / row.label / ('HCF' if row.hcf else 'standard')
    folder.mkdir(parents=True, exist_ok=True)
    im.save(folder / f'{i:05d}_original.png')
    for k in range(AUG_PER_IMAGE):
        augment_variant(im, k).save(folder / f'{i:05d}_aug{k+1:02d}.png')
print(f'각 분할 원본 1장당 총 {AUG_PER_IMAGE+1}장 저장 완료')


[전체 15.0%] 분할별 증강 및 저장 시작 (총 678)
[전체 15.0%] 분할별 증강 및 저장: 1/678 (0%)
[전체 18.0%] 분할별 증강 및 저장: 68/678 (10%)
[전체 21.0%] 분할별 증강 및 저장: 136/678 (20%)
[전체 24.0%] 분할별 증강 및 저장: 204/678 (30%)
[전체 27.0%] 분할별 증강 및 저장: 272/678 (40%)
[전체 30.0%] 분할별 증강 및 저장: 339/678 (50%)
[전체 33.0%] 분할별 증강 및 저장: 407/678 (60%)
[전체 36.0%] 분할별 증강 및 저장: 475/678 (70%)
[전체 39.0%] 분할별 증강 및 저장: 543/678 (80%)
[전체 42.0%] 분할별 증강 및 저장: 611/678 (90%)
[전체 45.0%] 분할별 증강 및 저장: 678/678 (100%)
각 분할 원본 1장당 총 30장 저장 완료


## 7. 저장된 증강 이미지 다시 불러오기

Drive에서 train·val·test 저장본을 로컬 임시 폴더로 복사해 학습과 평가에 사용합니다.

In [7]:
# Drive의 분할별 증강본을 다시 읽어 로컬에 복사
aug_files = list(saved.rglob('*.png'))
for src in progress(aug_files,45,60,'저장된 증강 이미지 다시 불러오기'):
    dst = LOCAL / src.relative_to(saved)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src,dst)
print('증강본 저장:', saved)


[전체 45.0%] 저장된 증강 이미지 다시 불러오기 시작 (총 20340)
[전체 45.0%] 저장된 증강 이미지 다시 불러오기: 1/20340 (0%)
[전체 46.5%] 저장된 증강 이미지 다시 불러오기: 2034/20340 (10%)
[전체 48.0%] 저장된 증강 이미지 다시 불러오기: 4068/20340 (20%)
[전체 49.5%] 저장된 증강 이미지 다시 불러오기: 6102/20340 (30%)
[전체 51.0%] 저장된 증강 이미지 다시 불러오기: 8136/20340 (40%)
[전체 52.5%] 저장된 증강 이미지 다시 불러오기: 10170/20340 (50%)
[전체 54.0%] 저장된 증강 이미지 다시 불러오기: 12204/20340 (60%)
[전체 55.5%] 저장된 증강 이미지 다시 불러오기: 14238/20340 (70%)
[전체 57.0%] 저장된 증강 이미지 다시 불러오기: 16272/20340 (80%)
[전체 58.5%] 저장된 증강 이미지 다시 불러오기: 18306/20340 (90%)
[전체 60.0%] 저장된 증강 이미지 다시 불러오기: 20340/20340 (100%)
증강본 저장: /content/drive/MyDrive/SEM_CNN_3epoch/20261108_081105/augmented_dataset


## 8. 데이터 로더 준비

이미지와 정답을 모델에 전달할 데이터 로더를 준비합니다.

In [8]:
norm=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class SemData(Dataset):
    def __init__(self,split):
        self.items=[(p,CLASSES.index(p.parent.parent.name),p.parent.name=='HCF') for p in sorted((LOCAL/split).rglob('*.png'))]
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,y,h=self.items[i]
        with Image.open(p) as im: x=norm(im.convert('RGB'))
        return x,y,h
sets={s:SemData(s) for s in ['train','val','test']}
loaders={s:DataLoader(d,batch_size=BATCH_SIZE,shuffle=(s=='train'),num_workers=0,pin_memory=device.type=='cuda') for s,d in sets.items()}
print({s:len(d) for s,d in sets.items()})


{'train': 14220, 'val': 4080, 'test': 2040}


## 9. 모델 설정

공통 특징 추출부를 고정하고 파단 유형 분류기와 HCF 여부 분류기를 함께 학습합니다. HCF 여부는 폴더 또는 파일명의 HCF 태그로 학습합니다.

In [9]:
# 공통 특징 추출부를 고정하고 두 분류기를 학습합니다.
model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad=False
model.fc=torch.nn.Sequential(torch.nn.Dropout(.3),torch.nn.Linear(model.fc.in_features,3))
model.to(device)
hcf_head=torch.nn.Linear(512,2).to(device)
features={}
def capture_fc_input(module, args): features['x']=args[0]
feature_hook=model.fc.register_forward_pre_hook(capture_fc_input)
counts=np.bincount([y for _,y,_ in sets['train'].items],minlength=3)
weights=torch.tensor(len(sets['train'])/(3*counts),dtype=torch.float32,device=device)
hcf_counts=np.bincount([int(h) for _,_,h in sets['train'].items],minlength=2)
if np.any(hcf_counts==0): raise ValueError('학습 분할에 HCF와 일반 사진이 각각 1장 이상 있어야 합니다.')
hcf_weights=torch.tensor(len(sets['train'])/(2*hcf_counts),dtype=torch.float32,device=device)
loss_fn=torch.nn.CrossEntropyLoss(weight=weights)
hcf_loss_fn=torch.nn.CrossEntropyLoss(weight=hcf_weights)
optimizer=torch.optim.AdamW(list(model.fc.parameters())+list(hcf_head.parameters()),lr=.001,weight_decay=.01)
print('학습 사진: 일반/HCF', hcf_counts)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 118MB/s]


학습 사진: 일반/HCF [11550  2670]


## 10. 20 epoch 학습

Train으로 학습하고 증강된 Validation 세트로 최적 모델을 선택합니다. 증강본은 같은 원본의 변형이므로 서로 독립적인 시편으로 해석하지 마세요.

In [10]:
best=float('inf'); history=[]
for epoch in range(EPOCHS):
    model.eval(); model.fc.train(); hcf_head.train()
    total=correct=hc=loss_sum=0
    for x,y,h in progress(loaders['train'],60+epoch*10,68+epoch*10,f'Epoch {epoch+1}/{EPOCHS} 학습'):
        x,y,h=x.to(device),y.to(device),h.to(device).long()
        optimizer.zero_grad(set_to_none=True)
        z=model(x); hz=hcf_head(features['x'])
        loss=loss_fn(z,y)+hcf_loss_fn(hz,h)
        loss.backward(); optimizer.step()
        total+=len(y); correct+=(z.argmax(1)==y).sum().item(); hc+=(hz.argmax(1)==h).sum().item(); loss_sum+=loss.item()*len(y)
    model.eval(); hcf_head.eval(); vl=vc=vh=vn=0
    with torch.no_grad():
        for x,y,h in progress(loaders['val'],68+epoch*10,70+epoch*10,f'Epoch {epoch+1}/{EPOCHS} 검증'):
            x,y,h=x.to(device),y.to(device),h.to(device).long()
            z=model(x); hz=hcf_head(features['x']); vl+=(loss_fn(z,y)+hcf_loss_fn(hz,h)).item()*len(y)
            vc+=(z.argmax(1)==y).sum().item(); vh+=(hz.argmax(1)==h).sum().item(); vn+=len(y)
    row={'epoch':epoch+1,'train_loss':loss_sum/total,'train_acc':correct/total,'train_hcf_acc':hc/total,'val_loss':vl/vn,'val_acc':vc/vn,'val_hcf_acc':vh/vn}
    history.append(row); print(row)
    if vl/vn<best:
        best=vl/vn; torch.save({'state_dict':model.state_dict(),'hcf_state_dict':hcf_head.state_dict(),'classes':CLASSES,'image_size':IMG_SIZE,'epoch':epoch+1},RUN/'best_model.pt')
pd.DataFrame(history).to_csv(RUN/'history.csv',index=False)


[전체 60.0%] Epoch 1/20 학습 시작 (총 445)
[전체 60.0%] Epoch 1/20 학습: 1/445 (0%)
[전체 60.8%] Epoch 1/20 학습: 45/445 (10%)
[전체 61.6%] Epoch 1/20 학습: 89/445 (20%)
[전체 62.4%] Epoch 1/20 학습: 134/445 (30%)
[전체 63.2%] Epoch 1/20 학습: 178/445 (40%)
[전체 64.0%] Epoch 1/20 학습: 223/445 (50%)
[전체 64.8%] Epoch 1/20 학습: 267/445 (60%)
[전체 65.6%] Epoch 1/20 학습: 312/445 (70%)
[전체 66.4%] Epoch 1/20 학습: 356/445 (80%)
[전체 67.2%] Epoch 1/20 학습: 401/445 (90%)
[전체 68.0%] Epoch 1/20 학습: 445/445 (100%)
[전체 68.0%] Epoch 1/20 검증 시작 (총 128)
[전체 68.0%] Epoch 1/20 검증: 1/128 (1%)
[전체 68.2%] Epoch 1/20 검증: 13/128 (10%)
[전체 68.4%] Epoch 1/20 검증: 26/128 (20%)
[전체 68.6%] Epoch 1/20 검증: 39/128 (30%)
[전체 68.8%] Epoch 1/20 검증: 52/128 (41%)
[전체 69.0%] Epoch 1/20 검증: 64/128 (50%)
[전체 69.2%] Epoch 1/20 검증: 77/128 (60%)
[전체 69.4%] Epoch 1/20 검증: 90/128 (70%)
[전체 69.6%] Epoch 1/20 검증: 103/128 (80%)
[전체 69.8%] Epoch 1/20 검증: 116/128 (91%)
[전체 70.0%] Epoch 1/20 검증: 128/128 (100%)
{'epoch': 1, 'train_loss': 0.7917814431478873, 'train_acc': 0

## 10-1. Python에서 불러올 모델 패키지 저장

학습이 완료된 후 실행하면 학습 결과 폴더의 `python_package` 안에 모델 파일과 `fiber_sem_model.py`, 사용 안내를 저장합니다. 다른 환경에 공유할 때는 이 세 파일을 모두 전달하세요. 이 단계는 모델을 추가 학습하지 않습니다.

In [11]:
# 현재 학습에서 최적으로 저장된 모델과 Python 모듈을 함께 저장
from pathlib import Path
import shutil
package_dir = RUN / 'python_package'
package_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = RUN / 'best_model.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError('best_model.pt가 없습니다. 먼저 10번 학습 셀을 실행하세요.')
module_source = '"""Load a SEM_CNN_30x_HCF checkpoint and classify an image.\n\nDependencies: torch, torchvision, Pillow. A checkpoint is required separately.\nOnly load checkpoint files from a trusted source.\n"""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torchvision import models, transforms\nfrom PIL import Image, ImageOps\n\n\nclass FiberSEMClassifier:\n    """Predict fracture class and, if present, HCF versus standard fiber."""\n    def __init__(self, checkpoint_path, device=None):\n        self.path = Path(checkpoint_path)\n        if not self.path.is_file():\n            raise FileNotFoundError(f\'Model checkpoint not found: {self.path}\')\n        self.device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n        checkpoint = torch.load(self.path, map_location=self.device, weights_only=True)\n        self.classes = list(checkpoint[\'classes\'])\n        self.image_size = int(checkpoint.get(\'image_size\', 224))\n        self.model = models.resnet18(weights=None)\n        self.model.fc = nn.Sequential(nn.Dropout(.3), nn.Linear(self.model.fc.in_features, len(self.classes)))\n        self.model.load_state_dict(checkpoint[\'state_dict\'])\n        self.model.to(self.device).eval()\n        self.hcf_head = None\n        if \'hcf_state_dict\' in checkpoint:\n            self.hcf_head = nn.Linear(512, 2).to(self.device)\n            self.hcf_head.load_state_dict(checkpoint[\'hcf_state_dict\'])\n            self.hcf_head.eval()\n        self.preprocess = transforms.Compose([\n            transforms.Resize((self.image_size, self.image_size)), transforms.ToTensor(),\n            transforms.Normalize([.485, .456, .406], [.229, .224, .225]),\n        ])\n\n    def predict(self, image_path):\n        """Return predictions and scores. Accepts a file path or PIL Image."""\n        if isinstance(image_path, Image.Image):\n            im = ImageOps.exif_transpose(image_path).convert(\'RGB\')\n        else:\n            with Image.open(image_path) as source:\n                im = ImageOps.exif_transpose(source).convert(\'RGB\')\n        x = self.preprocess(im).unsqueeze(0).to(self.device)\n        with torch.inference_mode():\n            m = self.model\n            x = m.conv1(x); x = m.bn1(x); x = m.relu(x); x = m.maxpool(x)\n            x = m.layer1(x); x = m.layer2(x); x = m.layer3(x); x = m.layer4(x)\n            features = torch.flatten(m.avgpool(x), 1)\n            scores = torch.softmax(m.fc(features), dim=1)[0].cpu().tolist()\n            best = max(range(len(scores)), key=scores.__getitem__)\n            result = {\'fracture_type\': self.classes[best],\n                      \'fracture_scores\': dict(zip(self.classes, scores)),\n                      \'fiber_type\': None, \'hcf_scores\': None}\n            if self.hcf_head is not None:\n                scores_hcf = torch.softmax(self.hcf_head(features), dim=1)[0].cpu().tolist()\n                result[\'fiber_type\'] = \'HCF\' if scores_hcf[1] > scores_hcf[0] else \'standard\'\n                result[\'hcf_scores\'] = {\'standard\': scores_hcf[0], \'HCF\': scores_hcf[1]}\n        return result\n'
(package_dir / 'fiber_sem_model.py').write_text(module_source, encoding='utf-8')
shutil.copy2(checkpoint_path, package_dir / 'best_model.pt')
readme = """필요 패키지: torch, torchvision, Pillow

fiber_sem_model.py와 best_model.pt를 같은 폴더에 두고 해당 폴더에서 실행:

from fiber_sem_model import FiberSEMClassifier
model = FiberSEMClassifier('best_model.pt')
result = model.predict('새로운_SEM_사진.png')
print(result)

다른 폴더의 코드에서 사용한다면 sys.path에 이 패키지 폴더를 추가하고
best_model.pt의 절대 경로를 FiberSEMClassifier에 전달하세요.
이 모델은 이미지를 학습한 가중치만 포함하며 학습 원본 사진은 포함하지 않습니다.
점수는 보정된 확률이 아닙니다. 신뢰할 수 있는 모델 파일만 여세요.
"""
(package_dir / 'README.txt').write_text(readme, encoding='utf-8')
print('Python 패키지 저장:', package_dir)
print('파일:', *[p.name for p in package_dir.iterdir()], sep='\n')


Python 패키지 저장: /content/drive/MyDrive/SEM_CNN_3epoch/20261108_081105/python_package
파일:
fiber_sem_model.py
best_model.pt
README.txt


## 11. 테스트 및 결과 저장

Test 전체(원본과 증강본) 결과와 Test 원본만의 결과를 각각 저장합니다. 실제 성능 판단에는 원본만의 결과를 우선 확인하세요.

In [12]:
# 최적 검증 손실 모델을 고정된 test에서 1회 평가
checkpoint=torch.load(RUN/'best_model.pt',map_location=device,weights_only=True); model.load_state_dict(checkpoint['state_dict']); hcf_head.load_state_dict(checkpoint['hcf_state_dict']); model.eval(); hcf_head.eval()
truth=[]; pred=[]; hcf=[]; hcf_pred=[]
with torch.no_grad():
    for x,y,h in progress(loaders['test'],90,99,'테스트 평가'):
        truth.extend(y.tolist()); pred.extend(model(x.to(device)).argmax(1).cpu().tolist()); hcf.extend(h.tolist()); hcf_pred.extend(hcf_head(features['x']).argmax(1).cpu().tolist())
hcf_report=classification_report(hcf,hcf_pred,labels=[0,1],target_names=['standard','HCF'],zero_division=0)
print('HCF 구분 결과:', hcf_report)
(RUN/'hcf_test_report.txt').write_text(hcf_report,encoding='utf-8')
report=classification_report(truth,pred,labels=list(range(3)),target_names=CLASSES,zero_division=0)
print('선택 epoch:',checkpoint['epoch']); print(report)
cm=confusion_matrix(truth,pred,labels=list(range(3)))
print('행=실제, 열=예측\n',cm)
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(RUN/'test_confusion_matrix.csv')
results=pd.DataFrame({'path':[str(p) for p,_,_ in sets['test'].items], 'true':[CLASSES[i] for i in truth],'pred':[CLASSES[i] for i in pred],'hcf':hcf,'hcf_pred':hcf_pred})
results.to_csv(RUN/'test_predictions.csv',index=False,encoding='utf-8-sig')
for flag in [False,True]:
    part=results[results.hcf==flag]
    print(('HCF' if flag else '일반'), '개수',len(part),'정확도',round((part['true']==part['pred']).mean(),3) if len(part) else '표본 없음')
(RUN/'test_report.txt').write_text(report,encoding='utf-8')
# 증강하지 않은 테스트 원본만 별도 평가
original_results = results[results['path'].str.endswith('_original.png')].copy()
if len(original_results):
    original_report = classification_report(original_results['true'], original_results['pred'], labels=CLASSES, target_names=CLASSES, zero_division=0)
    original_hcf_report = classification_report(original_results['hcf'].astype(int), original_results['hcf_pred'].astype(int), labels=[0,1], target_names=['standard','HCF'], zero_division=0)
    print('\n테스트 원본만 — 파단 유형:', len(original_results), '장\n', original_report)
    print('테스트 원본만 — HCF 구분:\n', original_hcf_report)
    (RUN/'test_original_report.txt').write_text(original_report,encoding='utf-8')
    (RUN/'hcf_test_original_report.txt').write_text(original_hcf_report,encoding='utf-8')
    pd.DataFrame(confusion_matrix(original_results['true'],original_results['pred'],labels=CLASSES),index=CLASSES,columns=CLASSES).to_csv(RUN/'test_original_confusion_matrix.csv')
    original_results.to_csv(RUN/'test_original_predictions.csv',index=False,encoding='utf-8-sig')
else:
    print('테스트 원본을 찾지 못했습니다. 저장 파일을 확인하세요.')

print('결과:',RUN)
print('[전체 100.0%] 모든 작업 완료',flush=True)
feature_hook.remove()
shutil.rmtree(LOCAL,ignore_errors=True)


[전체 90.0%] 테스트 평가 시작 (총 64)
[전체 90.1%] 테스트 평가: 1/64 (2%)
[전체 91.0%] 테스트 평가: 7/64 (11%)
[전체 91.8%] 테스트 평가: 13/64 (20%)
[전체 92.8%] 테스트 평가: 20/64 (31%)
[전체 93.7%] 테스트 평가: 26/64 (41%)
[전체 94.5%] 테스트 평가: 32/64 (50%)
[전체 95.5%] 테스트 평가: 39/64 (61%)
[전체 96.3%] 테스트 평가: 45/64 (70%)
[전체 97.3%] 테스트 평가: 52/64 (81%)
[전체 98.2%] 테스트 평가: 58/64 (91%)
[전체 99.0%] 테스트 평가: 64/64 (100%)
HCF 구분 결과:               precision    recall  f1-score   support

    standard       1.00      0.98      0.99      1830
         HCF       0.88      1.00      0.93       210

    accuracy                           0.99      2040
   macro avg       0.94      0.99      0.96      2040
weighted avg       0.99      0.99      0.99      2040

선택 epoch: 17
              precision    recall  f1-score   support

     bending       0.84      0.85      0.84       450
    cleaving       0.90      0.84      0.87      1230
    fracture       0.68      0.83      0.75       360

    accuracy                           0.84      2040
   macro a

## 12. 새 SEM 이미지 분류 및 주목 영역 표시

학습된 모델이 저장되어 있다면 이 셀만 따로 실행할 수 있습니다. 새 이미지를 한 장 또는 여러 장 업로드하세요. 기본적으로 가장 최근 실행에서 저장된 모델을 선택합니다. 다른 모델을 사용하려면 `MODEL_PATH`에 경로를 입력하세요.

빨간 박스는 Grad-CAM 점수가 가장 높은 위치 주변을 표시한 모델 주목 영역입니다. 파단 기점이나 실제 원인을 입증하지 않으며, 점수도 보정된 확률이 아닙니다.

해석 시 Bending은 mirror에서 mist·hackle로 이어지는 흔적과 기점을, Cleaving은 넓고 평탄한 면을 확인합니다. 이는 사람의 검토 기준이며 모델이 해당 특징을 검출하거나 우선순위로 학습했다는 뜻은 아닙니다.

In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import models, transforms
from google.colab import drive, files

drive.mount('/content/drive')
# 비워두면 가장 최근 저장된 모델을 선택합니다.
MODEL_PATH = ''
BOX_SIZE_RATIO = 0.25  # 표시 박스의 가로·세로 비율 (각각 이미지 크기의 25%)
if MODEL_PATH:
    model_path = Path(MODEL_PATH)
else:
    candidates = list(Path('/content/drive/MyDrive/SEM_CNN_3epoch').glob('*/best_model.pt'))
    if not candidates:
        raise FileNotFoundError('best_model.pt가 없습니다. 학습을 완료하거나 MODEL_PATH를 지정하세요.')
    model_path = max(candidates, key=lambda p: p.parent.name)
print('사용 모델:', model_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(model_path, map_location=device, weights_only=True)
classes = checkpoint['classes']
size = checkpoint.get('image_size', 224)
model = models.resnet18(weights=None)
model.fc = nn.Sequential(nn.Dropout(.3), nn.Linear(model.fc.in_features, len(classes)))
model.load_state_dict(checkpoint['state_dict'])
model.to(device).eval()
if 'hcf_state_dict' not in checkpoint:
    raise ValueError('이 모델에는 HCF 분류기가 없습니다. 업데이트된 노트북으로 새로 학습하세요.')
hcf_head = nn.Linear(512, 2).to(device)
hcf_head.load_state_dict(checkpoint['hcf_state_dict']); hcf_head.eval()
hcf_features = {}
def capture_hcf_features(module, args): hcf_features['x'] = args[0]
hcf_hook = model.fc.register_forward_pre_hook(capture_hcf_features)
preprocess = transforms.Compose([
    transforms.Resize((size, size)), transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

# 마지막 합성곱 층의 활성화와 기울기로 Grad-CAM 계산
activations = {}
def save_activation(module, inputs, output):
    activations['feature'] = output
    output.retain_grad()
hook = model.layer4[-1].register_forward_hook(save_activation)
result_dir = model_path.parent / 'new_predictions' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
result_dir.mkdir(parents=True, exist_ok=True)
print('분석할 SEM 사진을 한 장 또는 여러 장 선택하세요.')
uploaded = files.upload()
rows, reports = [], []
review_guide = {
    'bending': '검토 순서: 파괴 기점 및 mirror 영역 → 주변 mist·hackle 등 방사형 파면 흔적. 흔적이 모두 관찰되는 것은 아닙니다.',
    'cleaving': '검토 항목: 넓고 평탄한 절단면 및 주변 표면 흔적.',
    'fracture': '검토 항목: 파면의 거칠기, 기점 및 주변 손상. 유형 판별에는 추가 정보가 필요할 수 있습니다.',
}

try:
    for number, (filename, content) in enumerate(uploaded.items(), 1):
        try:
            with Image.open(BytesIO(content)) as source:
                image = ImageOps.exif_transpose(source).convert('RGB')
            x = preprocess(image).unsqueeze(0).to(device).requires_grad_(True)
            model.zero_grad(set_to_none=True)
            logits = model(x)
            probabilities = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
            hcf_scores = torch.softmax(hcf_head(hcf_features['x']), dim=1)[0].detach().cpu().numpy()
            hcf_label = 'HCF' if int(np.argmax(hcf_scores)) == 1 else '일반 광섬유'
            index = int(np.argmax(probabilities))
            logits[0, index].backward()
            feature = activations['feature']
            weights = feature.grad.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((weights * feature).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=(image.height, image.width), mode='bilinear', align_corners=False)[0, 0]
            cam = cam.detach().cpu().numpy()
            peak = float(cam.max())
            annotated = image.copy()
            if np.isfinite(peak) and peak > 1e-12:
                y, x_peak = np.unravel_index(np.argmax(cam), cam.shape)
                bw = max(1, int(image.width * BOX_SIZE_RATIO))
                bh = max(1, int(image.height * BOX_SIZE_RATIO))
                left = max(0, min(image.width-bw, int(x_peak)-bw//2))
                top = max(0, min(image.height-bh, int(y)-bh//2))
                ImageDraw.Draw(annotated).rectangle((left, top, left+bw-1, top+bh-1), outline='red', width=max(2, round(min(image.size)*0.006)))
                box_status = 'Grad-CAM 최대 지점 주변 표시'
            else:
                box_status = '유효한 주목 영역을 찾지 못해 박스 생략'
            output_image = result_dir / f'분석결과_{number:03d}.png'
            annotated.save(output_image)
            guide = review_guide.get(classes[index].lower(), '파면 특징을 별도로 확인하세요.')
            row = {'원본 파일명': filename, '예측 유형': classes[index], '박스 표시': box_status, 'HCF 예측': hcf_label, 'HCF 점수 (%)': round(float(hcf_scores[1])*100,2), '사람의 검토 참고 항목': guide,
                   **{f'{name} 점수 (%)': round(float(probabilities[i])*100, 2) for i, name in enumerate(classes)},
                   '결과 이미지': output_image.name}
            rows.append(row)
            reports.append(f"[{number}] {filename}\n예측: {classes[index]}\n" +
                           '\n'.join(f'{name}: {probabilities[i]*100:.2f}%' for i,name in enumerate(classes)) +
                           f'\n광섬유 구조 예측: {hcf_label} (HCF 점수 {hcf_scores[1]*100:.2f}%)\n{box_status}\n사람의 검토 참고 항목 (자동 검출 결과 아님): {guide}\n저장: {output_image.name}\n')
            plt.figure(figsize=(7, 6)); plt.imshow(annotated); plt.axis('off')
            plt.title(f'{filename}\nPrediction: {classes[index]} ({probabilities[index]*100:.1f}%)')
            plt.show(); print(row)
        except Exception as e:
            print(f'분석 실패: {filename} — {e}')
finally:
    hook.remove()
    hcf_hook.remove()

if rows:
    pd.DataFrame(rows).to_csv(result_dir/'분석결과.csv', index=False, encoding='utf-8-sig')
    report = 'SEM 이미지 분류 결과\n모델: ' + str(model_path) + '\n\n' + '\n'.join(reports)
    report += '\n빨간 박스는 모델 주목 영역이며 실제 파단 원인을 입증하지 않습니다. 검토 항목은 일반적인 참고 기준으로 사진에서 자동 검출된 특징이 아닙니다.\n'
    (result_dir/'분석결과.txt').write_text(report, encoding='utf-8')
    zip_path = result_dir.parent / (result_dir.name + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for item in sorted(result_dir.iterdir()):
            if item.is_file(): archive.write(item, arcname=item.name)
    print('결과 폴더:', result_dir)
    print('ZIP 파일:', zip_path)
    files.download(str(zip_path))
else:
    print('분석된 이미지가 없습니다.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
사용 모델: /content/drive/MyDrive/SEM_CNN_3epoch/20261108_081105/best_model.pt
분석할 SEM 사진을 한 장 또는 여러 장 선택하세요.


## 13. HTML용 모델 내보내기

학습 후 실행하면 `sem_model.onnx`를 Drive에 저장하고 PC로 다운로드합니다. HTML 화면에서 이 모델을 선택하세요.

In [ ]:
# HTML에서 사용할 ONNX 모델을 내보냅니다. 저장된 모델이 필요합니다.
import sys, subprocess
try:
    import onnx
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnx'])
import torch
from torch import nn
from torchvision import models
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
MODEL_PATH = ''  # 비워두면 가장 최근에 저장된 모델
candidates=list(Path('/content/drive/MyDrive/SEM_CNN_3epoch').glob('*/best_model.pt'))
if not MODEL_PATH and not candidates: raise FileNotFoundError('학습된 모델을 먼저 준비하세요.')
path=Path(MODEL_PATH) if MODEL_PATH else max(candidates,key=lambda p:p.parent.name)
ck=torch.load(path,map_location='cpu',weights_only=True)
if 'hcf_state_dict' not in ck: raise ValueError('HCF 모델이 없는 이전 체크포인트입니다. 다시 학습하세요.')
class ExportModel(nn.Module):
    def __init__(self):
        super().__init__()
        base=models.resnet18(weights=None)
        self.features=nn.Sequential(*list(base.children())[:-1])
        self.fracture=nn.Sequential(nn.Dropout(.3),nn.Linear(512,3))
        self.hcf=nn.Linear(512,2)
    def forward(self,x):
        x=self.features(x).flatten(1)
        return self.fracture(x),self.hcf(x)
m=ExportModel()
base=models.resnet18(weights=None)
base.fc=nn.Sequential(nn.Dropout(.3),nn.Linear(512,3))
base.load_state_dict(ck['state_dict'])
m.features.load_state_dict(nn.Sequential(*list(base.children())[:-1]).state_dict())
m.fracture.load_state_dict(base.fc.state_dict()); m.hcf.load_state_dict(ck['hcf_state_dict']); m.eval()
out=path.parent/'sem_model.onnx'
torch.onnx.export(m,torch.randn(1,3,224,224),str(out),input_names=['image'],output_names=['fracture','hcf'],opset_version=17,dynamo=False)
import onnx
onnx.checker.check_model(onnx.load(str(out)))
print('모델 저장:',out,'클래스 순서:',ck['classes'])
if ck['classes'] != ['bending','cleaving','fracture']: raise ValueError('HTML의 클래스 순서와 일치하지 않습니다.')
files.download(str(out))
